# Exp2 History Breakdown Plot-Only Viewer

This notebook does not rerun experiments. It reads the existing aggregated CSVs in `data/` and redraws the regular and broken-axis line plots.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)

from sigmod_exp_common import (
    TOL,
    apply_paper_style,
    current_run_stamp,
    ensure_dirs,
)

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_ivmh_history_breakdown').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

ALL_TABLE_SPECS = {
    'snap': {'label': 'SNAP'},
    'ivmh': {'label': 'IVMH'},
    'mono_wr': {'label': 'MONO-WR'},
    'dual_wr': {'label': 'DUAL-WR'},
    'epoch_wr': {'label': 'EPOCH-WR'},
}

CONFIG = {
    'structures': ['snap', 'ivmh', 'mono_wr', 'dual_wr', 'epoch_wr'],
    'sweeps': ['history', 'delta'],
    'broken_min_gap_ratio': 1.22,
    'broken_tick_step': 20,
    'broken_gap_hspace': 0.16,
    'broken_ylabel_x': 0.06,
    'broken_zigzag_amp': 0.018,
    'broken_zigzag_teeth': 18,
}

TABLE_LABELS = [ALL_TABLE_SPECS[key]['label'] for key in CONFIG['structures']]
RUN_STAMP = current_run_stamp()

SWEEP_XLABEL = {
    'history': 'Historical Scans (%)',
    'delta': 'Delta Scans (%)',
}
SWEEP_FILE_STEM = {
    'history': 'history',
    'delta': 'delta',
}
SERIES_STYLE = {
    'SNAP': (TOL['red'], ':', 'x'),
    'IVMH': (TOL['yellow'], '--', 'P'),
    'MONO-WR': (TOL['blue'], '-', 'o'),
    'DUAL-WR': (TOL['cyan'], '-', 's'),
    'EPOCH-WR': (TOL['green'], '-', 'D'),
}

print('EXP_DIR    :', EXP_DIR)
print('DATA_DIR   :', DATA_DIR)
print('FIGS_DIR   :', FIGS_DIR)
print('STRUCTURES :', TABLE_LABELS)
print('SWEEPS     :', CONFIG['sweeps'])
print('STAMP      :', RUN_STAMP)


In [ ]:
def latest_stack_path(sweep_type: str) -> Path:
    stable = DATA_DIR / f'ivmh-epoch-{sweep_type}-breakdown.csv'
    if stable.exists():
        return stable
    candidates = [
        path for path in DATA_DIR.glob(f'ivmh_epoch_{sweep_type}_breakdown_*.csv')
        if 'raw' not in path.name and 'progress' not in path.name
    ]
    if not candidates:
        raise FileNotFoundError(f'No stack CSV found for sweep={sweep_type}')
    return max(candidates, key=lambda path: path.stat().st_mtime)


def load_stack_df(sweep_type: str) -> pd.DataFrame:
    csv_path = latest_stack_path(sweep_type)
    df = pd.read_csv(csv_path)
    df = df[df['table_label'].isin(TABLE_LABELS)].copy()
    print(f'{sweep_type}: loaded', csv_path)
    display(df.head())
    return df


STACK_RESULTS = {sweep_type: load_stack_df(sweep_type) for sweep_type in CONFIG['sweeps']}


In [ ]:
def build_series_legend_handles(labels):
    handles = []
    for label in labels:
        color, linestyle, marker = SERIES_STYLE[label]
        handles.append(
            Line2D([0], [0], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5.5)
        )
    return handles, labels


def group_totals(df: pd.DataFrame, labels):
    sweep_values = sorted(df['special_scans'].drop_duplicates().tolist())
    totals = {}
    for label in labels:
        pivot = (
            df[df['table_label'] == label]
            .pivot(index='special_scans', columns='component', values='duration_ms')
            .reindex(index=sweep_values, fill_value=0.0)
            .fillna(0.0)
        )
        totals[label] = pivot.sum(axis=1).tolist()
    sweep_pcts = sorted(df['sweep_pct'].drop_duplicates().tolist())
    return sweep_values, sweep_pcts, totals


def apply_axis_font_style(ax):
    return ax


def render_regular_plot(df: pd.DataFrame, sweep_type: str):
    labels = [label for label in TABLE_LABELS if label in set(df['table_label'])]
    sweep_values, sweep_pcts, totals = group_totals(df, labels)
    sweep_pct_labels = [f'{value:g}' for value in sweep_pcts]
    handles, legend_labels = build_series_legend_handles(labels)

    fig, ax = plt.subplots(figsize=(7.6, 3.9))
    for label in labels:
        color, linestyle, marker = SERIES_STYLE[label]
        ax.plot(
            sweep_pcts,
            totals[label],
            color=color,
            linestyle=linestyle,
            marker=marker,
            linewidth=1.8,
            markersize=5.5,
            label=label,
        )

    ax.set_xlabel(SWEEP_XLABEL[sweep_type])
    ax.set_ylabel('Total (ms / tx)')
    ax.set_xticks(sweep_pcts)
    ax.set_xticklabels(sweep_pct_labels)
    ax.set_ylim(bottom=0)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    apply_axis_font_style(ax)
    ax.legend(handles, legend_labels, loc='upper left', framealpha=0.95, ncol=3)
    fig.tight_layout()

    stem = SWEEP_FILE_STEM[sweep_type]
    stamped_pdf = FIGS_DIR / f'exp2-{stem}-breakdown-compare-view-{RUN_STAMP}.pdf'
    latest_pdf = FIGS_DIR / f'exp2-{stem}-breakdown-compare-view.pdf'
    stamped_png = FIGS_DIR / f'exp2-{stem}-breakdown-compare-view-{RUN_STAMP}.png'
    latest_png = FIGS_DIR / f'exp2-{stem}-breakdown-compare-view.png'
    fig.savefig(stamped_pdf, format='pdf', bbox_inches='tight')
    fig.savefig(latest_pdf, format='pdf', bbox_inches='tight')
    fig.savefig(stamped_png, dpi=220, bbox_inches='tight')
    fig.savefig(latest_png, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved', stamped_pdf)
    print('Saved', latest_pdf)
    print('Saved', stamped_png)
    print('Saved', latest_png)


for sweep_type in CONFIG['sweeps']:
    render_regular_plot(STACK_RESULTS[sweep_type], sweep_type)


In [ ]:
def draw_axis_break(fig, ax_top, ax_bottom, amp=0.018, teeth=18):
    pos_top = ax_top.get_position()
    pos_bottom = ax_bottom.get_position()
    amp_fig = amp * min(pos_top.height, pos_bottom.height)
    amp_top = amp_fig / max(pos_top.height, 1e-6)
    amp_bottom = amp_fig / max(pos_bottom.height, 1e-6)
    xs = [i / teeth for i in range(teeth + 1)]
    ys_top = [(-amp_top if i % 2 == 0 else amp_top) for i in range(teeth + 1)]
    ys_bottom = [(1 - amp_bottom if i % 2 == 0 else 1 + amp_bottom) for i in range(teeth + 1)]
    line_kwargs = dict(color='black', clip_on=False, linewidth=plt.rcParams.get('axes.linewidth', 0.8))
    ax_top.plot(xs, ys_top, transform=ax_top.transAxes, **line_kwargs)
    ax_bottom.plot(xs, ys_bottom, transform=ax_bottom.transAxes, **line_kwargs)


def hide_broken_edge_gridlines(ax_top, ax_bottom):
    top_lines = ax_top.get_ygridlines()
    bottom_lines = ax_bottom.get_ygridlines()
    if top_lines:
        top_lines[0].set_visible(False)
    if bottom_lines:
        bottom_lines[-1].set_visible(False)


def render_broken_plot(df: pd.DataFrame, sweep_type: str):
    labels = [label for label in TABLE_LABELS if label in set(df['table_label'])]
    sweep_values, sweep_pcts, totals = group_totals(df, labels)
    sweep_pct_labels = [f'{value:g}' for value in sweep_pcts]

    if 'SNAP' not in totals:
        print(f'{sweep_type}: no SNAP series, skip broken plot')
        return

    non_snap_labels = [label for label in labels if label != 'SNAP']
    non_snap_max = max((max(totals[label]) for label in non_snap_labels), default=0.0)
    snap_min = min(totals['SNAP'])
    snap_max = max(totals['SNAP'])

    if not (non_snap_max > 0 and snap_min > non_snap_max * CONFIG['broken_min_gap_ratio']):
        print(f'{sweep_type}: SNAP separation is not large enough for broken-axis replot.')
        return

    bottom_max = non_snap_max * 1.10
    top_min = max(snap_min * 0.94, bottom_max + max(1.0, non_snap_max * 0.08))
    top_max = snap_max * 1.05

    bottom_range = max(bottom_max, 1.0)
    top_range = max(top_max - top_min, 1.0)
    fig = plt.figure(figsize=(7.2, 4.9))
    gs = fig.add_gridspec(2, 1, height_ratios=[top_range, bottom_range], hspace=CONFIG['broken_gap_hspace'])
    ax_top = fig.add_subplot(gs[0])
    ax_bottom = fig.add_subplot(gs[1], sharex=ax_top)

    for label in labels:
        color, linestyle, marker = SERIES_STYLE[label]
        ax_top.plot(
            sweep_pcts,
            totals[label],
            color=color,
            linestyle=linestyle,
            marker=marker,
            linewidth=1.8,
            markersize=5.5,
            label=label,
        )
        ax_bottom.plot(
            sweep_pcts,
            totals[label],
            color=color,
            linestyle=linestyle,
            marker=marker,
            linewidth=1.8,
            markersize=5.5,
            label=label,
        )

    ax_bottom.set_ylim(0, bottom_max)
    ax_top.set_ylim(top_min, top_max)
    ax_bottom.yaxis.set_major_locator(MultipleLocator(CONFIG['broken_tick_step']))
    ax_top.yaxis.set_major_locator(MultipleLocator(CONFIG['broken_tick_step']))

    ax_top.spines['bottom'].set_visible(False)
    ax_bottom.spines['top'].set_visible(False)
    ax_top.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    ax_bottom.tick_params(axis='x', which='both', top=False)
    ax_top.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax_bottom.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    hide_broken_edge_gridlines(ax_top, ax_bottom)
    ax_bottom.set_xlabel(SWEEP_XLABEL[sweep_type])
    ax_bottom.set_xticks(sweep_pcts)
    ax_bottom.set_xticklabels(sweep_pct_labels)
    apply_axis_font_style(ax_top)
    apply_axis_font_style(ax_bottom)
    fig.supylabel('Total (ms / tx)', x=CONFIG['broken_ylabel_x'])
    ax_bottom.legend(loc='lower left', bbox_to_anchor=(0.01, 0.02), framealpha=0.95, ncol=2)
    draw_axis_break(fig, ax_top, ax_bottom, amp=CONFIG['broken_zigzag_amp'], teeth=CONFIG['broken_zigzag_teeth'])
    fig.tight_layout(rect=[0.08, 0.02, 1, 1])

    stem = SWEEP_FILE_STEM[sweep_type]
    stamped_pdf = FIGS_DIR / f'exp2-{stem}-breakdown-compare-broken-view-{RUN_STAMP}.pdf'
    latest_pdf = FIGS_DIR / f'exp2-{stem}-breakdown-compare-broken-view.pdf'
    stamped_png = FIGS_DIR / f'exp2-{stem}-breakdown-compare-broken-view-{RUN_STAMP}.png'
    latest_png = FIGS_DIR / f'exp2-{stem}-breakdown-compare-broken-view.png'
    fig.savefig(stamped_pdf, format='pdf', bbox_inches='tight')
    fig.savefig(latest_pdf, format='pdf', bbox_inches='tight')
    fig.savefig(stamped_png, dpi=220, bbox_inches='tight')
    fig.savefig(latest_png, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved', stamped_pdf)
    print('Saved', latest_pdf)
    print('Saved', stamped_png)
    print('Saved', latest_png)


for sweep_type in CONFIG['sweeps']:
    render_broken_plot(STACK_RESULTS[sweep_type], sweep_type)
